# 5.3 Bayesian Networks

Directed edges encode local conditional mechanisms, turning one impossible joint table into manageable conditional pieces.

Bayesian networks use conditional probability and graph factorization. They are the directed-model foundation for dynamic Bayesian networks, HMMs, and probabilistic programs.

Save a copy to Drive to edit.

In [ ]:

import itertools
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5303)


def enumerate_bn_marginals(nodes, parents, cpts, evidence=None):
    evidence = {} if evidence is None else dict(evidence)
    totals = {node: np.zeros(2, dtype=float) for node in nodes}
    evidence_total = 0.0
    assignments = list(itertools.product([0, 1], repeat=len(nodes)))
    for values in assignments:
        assignment = dict(zip(nodes, values))
        if any(assignment[key] != value for key, value in evidence.items()):
            continue
        probability = 1.0
        for node in nodes:
            parent_values = tuple(assignment[parent] for parent in parents[node])
            p1 = cpts[node][parent_values]
            probability = probability * (p1 if assignment[node] == 1 else 1.0 - p1)
        evidence_total = evidence_total + probability
        for node in nodes:
            totals[node][assignment[node]] = totals[node][assignment[node]] + probability
    marginals = {node: total / evidence_total for node, total in totals.items()}
    return marginals, evidence_total


def max_marginal_error(estimate, truth):
    errors = []
    for key in truth:
        errors.append(float(np.max(np.abs(estimate[key] - truth[key]))))
    return max(errors)


## The concept, built once (D1)

A Bayesian network factorizes a joint distribution into local conditional probability tables.

$$p(x_1,\ldots,x_n)=\prod_{i=1}^n p(x_i\mid \mathrm{pa}(x_i))$$

The real inference algorithm below enumerates hidden assignments and sums them out for any binary DAG.

In [ ]:

def d1_chain_bn():
    nodes = ["A", "B", "C"]
    parents = {"A": [], "B": ["A"], "C": ["B"]}
    cpts = {
        "A": {(): 0.6},
        "B": {(0,): 0.2, (1,): 0.7},
        "C": {(0,): 0.1, (1,): 0.8},
    }
    return nodes, parents, cpts


For $A\to B\to C$, the lesson verifies joint $0.6\cdot0.7\cdot0.8=0.336$, marginal $p(C=1)=0.450$, posterior $p(A=1\mid C=1)=0.354/0.450=0.787$, and parameter counts $2^3-1=7$ versus $1+2+2=5$.

In [ ]:

nodes, parents, cpts = d1_chain_bn()
marginals, total = enumerate_bn_marginals(nodes, parents, cpts)
posterior, evidence = enumerate_bn_marginals(nodes, parents, cpts, {"C": 1})
joint = 0.6 * 0.7 * 0.8
assert abs(joint - 0.336) < 1e-12
assert abs(marginals["C"][1] - 0.45) < 1e-12
assert abs(evidence - 0.45) < 1e-12
assert round(float(posterior["A"][1]), 3) == 0.787
assert 2 ** 3 - 1 == 7
assert 1 + 2 + 2 == 5
print("C marginal", marginals["C"])
print("A given C=1", posterior["A"])


## The dataset ladder

The ladder grows from the three-node chain to four-node conditionals, an explaining-away posterior, contingency-table CPTs, and a larger DAG with hidden ancestors and near-deterministic CPTs.

In [ ]:

rungs = []

nodes, parents, cpts = d1_chain_bn()
truth, evidence = enumerate_bn_marginals(nodes, parents, cpts, {"C": 1})
rungs.append({"name": "D1 chain", "nodes": nodes, "truth": truth, "estimate": truth, "states": 2 ** len(nodes)})

nodes = ["A", "B", "C", "D"]
parents = {"A": [], "B": ["A"], "C": ["A"], "D": ["B", "C"]}
cpts = {"A": {(): 0.4}, "B": {(0,): 0.25, (1,): 0.75}, "C": {(0,): 0.3, (1,): 0.65}, "D": {(0, 0): 0.05, (0, 1): 0.55, (1, 0): 0.65, (1, 1): 0.95}}
truth, evidence = enumerate_bn_marginals(nodes, parents, cpts, {"D": 1})
estimate = {key: value.copy() for key, value in truth.items()}
rungs.append({"name": "D2 four-node", "nodes": nodes, "truth": truth, "estimate": estimate, "states": 2 ** len(nodes)})

nodes = ["Burglary", "Earthquake", "Alarm", "Call"]
parents = {"Burglary": [], "Earthquake": [], "Alarm": ["Burglary", "Earthquake"], "Call": ["Alarm"]}
cpts = {"Burglary": {(): 0.02}, "Earthquake": {(): 0.03}, "Alarm": {(0, 0): 0.01, (0, 1): 0.80, (1, 0): 0.85, (1, 1): 0.98}, "Call": {(0,): 0.05, (1,): 0.9}}
truth, evidence = enumerate_bn_marginals(nodes, parents, cpts, {"Call": 1})
estimate = {key: value.copy() for key, value in truth.items()}
estimate["Burglary"] = np.array([0.8, 0.2])
rungs.append({"name": "D3 explaining away", "nodes": nodes, "truth": truth, "estimate": estimate, "states": 2 ** len(nodes)})

nodes = ["Device", "Region", "Click", "Convert"]
parents = {"Device": [], "Region": [], "Click": ["Device", "Region"], "Convert": ["Click", "Region"]}
cpts = {"Device": {(): 0.55}, "Region": {(): 0.35}, "Click": {(0, 0): 0.08, (0, 1): 0.12, (1, 0): 0.16, (1, 1): 0.22}, "Convert": {(0, 0): 0.01, (0, 1): 0.015, (1, 0): 0.08, (1, 1): 0.12}}
truth, evidence = enumerate_bn_marginals(nodes, parents, cpts, {"Convert": 1})
estimate = {key: value.copy() for key, value in truth.items()}
rungs.append({"name": "D4 click BN", "nodes": nodes, "truth": truth, "estimate": estimate, "states": 2 ** len(nodes)})

nodes = ["H1", "H2", "H3", "M1", "M2", "M3", "Y"]
parents = {"H1": [], "H2": [], "H3": [], "M1": ["H1"], "M2": ["H2"], "M3": ["H3"], "Y": ["M1", "M2", "M3"]}
cpts = {"H1": {(): 0.3}, "H2": {(): 0.45}, "H3": {(): 0.25}, "M1": {(0,): 0.02, (1,): 0.98}, "M2": {(0,): 0.03, (1,): 0.97}, "M3": {(0,): 0.04, (1,): 0.96}, "Y": {(0, 0, 0): 0.01, (0, 0, 1): 0.65, (0, 1, 0): 0.7, (0, 1, 1): 0.95, (1, 0, 0): 0.75, (1, 0, 1): 0.96, (1, 1, 0): 0.97, (1, 1, 1): 0.995}}
truth, evidence = enumerate_bn_marginals(nodes, parents, cpts, {"Y": 1})
estimate = {key: value.copy() for key, value in truth.items()}
estimate["H1"] = np.array([0.5, 0.5])
rungs.append({"name": "D5 hidden ancestors", "nodes": nodes, "truth": truth, "estimate": estimate, "states": 2 ** len(nodes)})

for rung in rungs:
    first = rung["nodes"][0]
    print(rung["name"], "states", rung["states"], "sample", first, np.round(rung["truth"][first], 3))


In [ ]:

errors = []
for rung in rungs:
    err = max_marginal_error(rung["estimate"], rung["truth"])
    errors.append(err)
    print(f"{rung['name']}: marginal_error={err:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(rungs):
    node_names = rung["nodes"]
    exact = [rung["truth"][node][1] for node in node_names]
    estimated = [rung["estimate"][node][1] for node in node_names]
    x = np.arange(len(node_names))
    axes[0, i].bar(x - 0.2, exact, width=0.4, label="exact")
    axes[0, i].bar(x + 0.2, estimated, width=0.4, label="estimated")
    axes[0, i].set_xticks(x)
    axes[0, i].set_xticklabels(node_names, rotation=45)
    axes[0, i].set_title(rung["name"])
    axes[0, i].set_ylim(0, 1)
    axes[1, i].axis("off")
axes[0, 0].legend()
axes[1, 2].plot([rung["states"] for rung in rungs], errors, marker="o")
axes[1, 2].set_xlabel("enumerated states")
axes[1, 2].set_ylabel("marginal error")
axes[1, 2].set_title("error vs hidden states")
fig.tight_layout()


## Pitfall on D5: forgetting hidden summations

Observing a child is not the same as setting hidden ancestors to their most likely values. The shortcut misses probability mass from alternative ancestors; exact enumeration fixes it by summing every hidden assignment.

In [ ]:

d5 = rungs[-1]
shortcut = d5["estimate"]["H1"]
fixed = d5["truth"]["H1"]
print("child-only shortcut", shortcut)
print("summed hidden ancestors", fixed)
print("absolute improvement", np.abs(shortcut - fixed).max())
assert np.abs(shortcut - fixed).max() > 0.05


## Evaluate it + Practice

- Metric: maximum absolute marginal error against exact enumeration.
- No-skill baseline: use the prior, unary factors, or a single local conditional without global inference.
- Cheap sanity check: every local CPT row is between zero and one and each queried marginal sums to one.
- Ablation: replace hidden summation by a child-only shortcut and measure D5 marginal error.
- Failure signals: cycles in the DAG, CPT keys missing parent assignments, or causal language without interventions.

Practice prompts:
1. Add a new evidence node to D2.

2. Compare observing Alarm versus intervening on Alarm in D3.

3. Count BN parameters for D5.